# Medical AI Chatbot — v2 Final (with Fixes)

All fixes are **integrated directly** into the relevant cells — no patching needed.

| Cell | Content | Fixes |
|------|---------|-------|
| 1 | Installation | — |
| 2 | Imports, Config, Model Load | — |
| 3 | Layer Definitions & Session State | — |
| 4 | Language Detection & State Machine | — |
| 5 | **LLMGenerator** | Fix 1, 2, 3 |
| 6 | **Medical Agent** | Fix 3 (session_id) |
| 7 | **LangGraph Multi-Agent** | Fix 4 |
| 8 | FastAPI Server & Launch | — |

## Cell 1 — Installation
Run once, then restart the kernel before continuing.

In [ ]:
# Cell 1 - Installation
# Run once then restart the kernel before Cell 2

import subprocess
subprocess.run(["pip","install","-q","--upgrade","transformers","optimum","auto-gptq"], check=False)
subprocess.run(["pip","install","-q","fastapi","uvicorn","pyngrok","nest_asyncio"], check=False)

print("Installation complete. Restart the kernel before continuing.")

## Cell 2 — Imports, Configuration, Report Storage & Model Load

In [ ]:
# Cell 2 — Imports, Configuration, Report Storage & Model Load

import os, re, json, uuid, copy, threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from enum import Enum
from contextlib import asynccontextmanager

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
import nest_asyncio
import uvicorn

nest_asyncio.apply()

# ── Model & Server ─────────────────────────────────────────────────────────────
MODEL_ID         = "fady-50/F-Chat-Model-GPTQ"
NGROK_AUTH_TOKEN = "38YIqmgsWHFwzD4ykQoFV4bhOuF_4houKhQZVgCSJncunqURL"
PORT             = 8000
USE_FP16         = True

# ── Symptom Detection ──────────────────────────────────────────────────────────
LOW_CONFIDENCE             = 0.25
MAX_FOLLOWUPS_WITH_SYMPTOM = 2

# ── Early Exit ─────────────────────────────────────────────────────────────────
# The model stops asking and auto-generates the report when evidence is strong enough.
# Rule A: one layer with confidence >= this value
EARLY_EXIT_CONFIDENCE      = 0.80
# Rule B: at least this many positive layers with average confidence >= below
MIN_LAYERS_AUTO_EXIT       = 2
MULTI_LAYER_AVG_CONFIDENCE = 0.60

# ── Semantic Detection ─────────────────────────────────────────────────────────
# LLM validates patient message when keywords find nothing.
# Below this: skip the layer. Above this: treat as matching.
SEMANTIC_THRESHOLD = 0.45

# ── Smart Jump ─────────────────────────────────────────────────────────────────
SMART_JUMP_MIN_SCORE = 2    # minimum keyword hits in the target layer
SMART_JUMP_MARGIN    = 1    # must beat current layer score by at least this

# ── Severity Weights ───────────────────────────────────────────────────────────
LAYER_SEVERITY_WEIGHT: Dict[str, float] = {
    "skin":0.5,"soft_tissue":0.7,"muscle_tendon":0.8,
    "joint":0.9,"bone":1.0,"nerve":1.1,"vascular":1.3,
}

CORRELATION_RULES = [
    ({"soft_tissue","nerve"},    "Possible nerve compression",               0.3),
    ({"soft_tissue","vascular"}, "Possible vascular compromise",             0.4),
    ({"bone","nerve"},           "Possible fracture with nerve involvement",  0.5),
    ({"muscle_tendon","joint"},  "Possible tendon-joint complex injury",      0.3),
    ({"nerve","vascular"},       "Possible neurovascular bundle injury",      0.6),
]

EMERGENCY_KEYWORDS = {
    "en": ["can't breathe","cannot breathe","chest pain","heart attack","stroke",
           "unconscious","passing out","bleeding heavily","severe bleeding",
           "paralyzed","can't move","not moving","stopped breathing","no pulse"],
    "ar": ["\u0644\u0627 \u0623\u0633\u062a\u0637\u064a\u0639 \u0627\u0644\u062a\u0646\u0641\u0633","\u0623\u0644\u0645 \u0641\u064a \u0627\u0644\u0635\u062f\u0631",
           "\u0646\u0648\u0628\u0629 \u0642\u0644\u0628\u064a\u0629","\u062c\u0644\u0637\u0629","\u0641\u0627\u0642\u062f \u0627\u0644\u0648\u0639\u064a",
           "\u0646\u0632\u064a\u0641 \u0634\u062f\u064a\u062f","\u0644\u0627 \u064a\u062a\u062d\u0631\u0643","\u062a\u0648\u0642\u0641 \u0639\u0646 \u0627\u0644\u062a\u0646\u0641\u0633"],
}

# ── Report Storage ─────────────────────────────────────────────────────────────
REPORTS_DIR = "generated_reports"
os.makedirs(REPORTS_DIR, exist_ok=True)
reports_db: Dict[str, dict] = {}

def save_report_to_disk(session_id: str, report: dict):
    with open(os.path.join(REPORTS_DIR, f"{session_id}.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

def load_report_from_disk(session_id: str) -> Optional[dict]:
    path = os.path.join(REPORTS_DIR, f"{session_id}.json")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return None

# ── Model Load ─────────────────────────────────────────────────────────────────
print("Cleaning port 8000...")
try: os.system("fuser -k 8000/tcp")
except: pass

print("Loading model to GPU...")
global_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
global_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto",
    torch_dtype=torch.float16 if USE_FP16 else torch.float32,
    low_cpu_mem_usage=True, trust_remote_code=True,
)
global_model.eval()
print("Model ready.")

## Cell 3 — Layer Definitions & Session State

In [ ]:
# Cell 3 — Layer Definitions & Session State

class Layer(Enum):
    SKIN="skin"; SOFT_TISSUE="soft_tissue"; MUSCLE_TENDON="muscle_tendon"
    JOINT="joint"; BONE="bone"; NERVE="nerve"; VASCULAR="vascular"

LAYER_ORDER: List[Layer] = [
    Layer.SKIN,Layer.SOFT_TISSUE,Layer.MUSCLE_TENDON,
    Layer.JOINT,Layer.BONE,Layer.NERVE,Layer.VASCULAR,
]

# Clinical SOCRATES follow-up aspects per layer
# Used to guide the LLM when asking deeper questions
LAYER_SOCRATES: Dict[str, str] = {
    "skin":          "onset, spread pattern, associated itching or heat, any contact with irritants",
    "soft_tissue":   "tenderness on palpation, firmness, warmth, size change over time",
    "muscle_tendon": "pain on active vs passive movement, morning stiffness, weakness grade",
    "joint":         "range of motion, locking episodes, swelling pattern, morning vs evening",
    "bone":          "point tenderness, weight-bearing ability, deformity, trauma mechanism",
    "nerve":         "dermatomal distribution, sensory loss, motor weakness, aggravating positions",
    "vascular":      "claudication distance, color on elevation vs dependency, capillary refill time",
}

LAYER_META: Dict[Layer, dict] = {
    Layer.SKIN: {
        "description_en": "skin surface: rashes, wounds, color changes, temperature, itching, visible swelling",
        "description_ar": "\u0633\u0637\u062d \u0627\u0644\u062c\u0644\u062f: \u0637\u0641\u062d\u060c \u062c\u0631\u0648\u062d\u060c \u062a\u063a\u064a\u0631 \u0644\u0648\u0646\u060c \u062d\u0631\u0627\u0631\u0629\u060c \u062d\u0643\u0629\u060c \u062a\u0648\u0631\u0645 \u0638\u0627\u0647\u0631",
        "fallback_q_en": "Looking at the skin over the affected area — do you notice any redness, rash, wound, warmth, or swelling on the surface?",
        "fallback_q_ar": "\u0628\u0627\u0644\u0646\u0638\u0631 \u0625\u0644\u0649 \u062c\u0644\u062f \u0627\u0644\u0645\u0646\u0637\u0642\u0629 \u2014 \u0647\u0644 \u062a\u0644\u0627\u062d\u0638 \u0627\u062d\u0645\u0631\u0627\u0631\u0627\u064b \u0623\u0648 \u0637\u0641\u062d\u0627\u064b \u0623\u0648 \u062c\u0631\u062d\u0627\u064b \u0623\u0648 \u062f\u0641\u0621\u0627\u064b \u0623\u0648 \u062a\u0648\u0631\u0645\u0627\u064b \u0639\u0644\u0649 \u0627\u0644\u0633\u0637\u062d\u061f",
        "positive_keywords": {
            "en": ["rash","red","redness","wound","swelling","itch","hot","warm","blister","peel","lesion","sore","bruise","discolor","skin","scratch"],
            "ar": ["\u0637\u0641\u062d","\u0627\u062d\u0645\u0631\u0627\u0631","\u062c\u0631\u062d","\u062a\u0648\u0631\u0645","\u062d\u0643\u0629","\u062d\u0631\u0627\u0631\u0629","\u0628\u062b\u0648\u0631","\u0622\u0641\u0629","\u0642\u0631\u062d\u0629","\u062c\u0644\u062f","\u062e\u062f\u0634"],
        },
    },
    Layer.SOFT_TISSUE: {
        "description_en": "subcutaneous tissue: lumps, bruising, localised tenderness, deep soft swelling",
        "description_ar": "\u0623\u0646\u0633\u062c\u0629 \u062a\u062d\u062a \u0627\u0644\u062c\u0644\u062f: \u0643\u062a\u0644\u060c \u0643\u062f\u0645\u0627\u062a\u060c \u0623\u0644\u0645 \u0645\u0648\u0636\u0639\u064a\u060c \u062a\u0648\u0631\u0645 \u0639\u0645\u064a\u0642 \u0646\u0627\u0639\u0645",
        "fallback_q_en": "When you gently press on the area — do you feel a lump, unusual firmness, or tenderness beneath the skin surface?",
        "fallback_q_ar": "\u0639\u0646\u062f\u0645\u0627 \u062a\u0636\u063a\u0637 \u0628\u0631\u0641\u0642 \u0639\u0644\u0649 \u0627\u0644\u0645\u0646\u0637\u0642\u0629 \u2014 \u0647\u0644 \u062a\u0634\u0639\u0631 \u0628\u0643\u062a\u0644\u0629 \u0623\u0648 \u0635\u0644\u0627\u0628\u0629 \u063a\u064a\u0631 \u0637\u0628\u064a\u0639\u064a\u0629 \u0623\u0648 \u0623\u0644\u0645 \u062a\u062d\u062a \u0633\u0637\u062d \u0627\u0644\u062c\u0644\u062f\u061f",
        "positive_keywords": {
            "en": ["lump","bruise","tender","soft swelling","mass","bump","knot","puffy","deep pain","pressure pain"],
            "ar": ["\u0643\u062a\u0644\u0629","\u0643\u062f\u0645\u0629","\u0623\u0644\u0645 \u0639\u0646\u062f \u0627\u0644\u0636\u063a\u0637","\u062a\u0648\u0631\u0645 \u0646\u0627\u0639\u0645","\u0635\u0644\u0627\u0628\u0629","\u0648\u0631\u0645 \u0639\u0645\u064a\u0642"],
        },
    },
    Layer.MUSCLE_TENDON: {
        "description_en": "muscles and tendons: pain on movement, weakness, cramps, strain, tears",
        "description_ar": "\u0639\u0636\u0644\u0627\u062a \u0648\u0623\u0648\u062a\u0627\u0631: \u0623\u0644\u0645 \u0639\u0646\u062f \u0627\u0644\u062d\u0631\u0643\u0629\u060c \u0636\u0639\u0641\u060c \u062a\u0634\u0646\u062c\u0627\u062a\u060c \u0625\u062c\u0647\u0627\u062f",
        "fallback_q_en": "When you actively use or move the area — does the pain worsen? Do you feel any muscle weakness, tightness, or cramping?",
        "fallback_q_ar": "\u0639\u0646\u062f\u0645\u0627 \u062a\u062d\u0631\u0643 \u0627\u0644\u0645\u0646\u0637\u0642\u0629 \u0623\u0648 \u062a\u0633\u062a\u062e\u062f\u0645\u0647\u0627 \u2014 \u0647\u0644 \u064a\u0632\u062f\u0627\u062f \u0627\u0644\u0623\u0644\u0645\u061f \u0647\u0644 \u062a\u0634\u0639\u0631 \u0628\u0636\u0639\u0641 \u0627\u0644\u0639\u0636\u0644\u0629 \u0623\u0648 \u0634\u062f\u0647\u0627 \u0623\u0648 \u062a\u0634\u0646\u062c\u0647\u0627\u061f",
        "positive_keywords": {
            "en": ["weak","cramp","strain","tear","spasm","tightness","pain with movement","cannot lift","muscle","pull","knot","ache"],
            "ar": ["\u0636\u0639\u0641","\u062a\u0634\u0646\u062c","\u0625\u062c\u0647\u0627\u062f","\u062a\u0645\u0632\u0642","\u0634\u062f","\u0623\u0644\u0645 \u0639\u0646\u062f \u0627\u0644\u062d\u0631\u0643\u0629","\u0639\u0636\u0644\u0629","\u0634\u062f \u0639\u0636\u0644\u064a"],
        },
    },
    Layer.JOINT: {
        "description_en": "joints: range of motion, clicking, stiffness, deep aching, instability, locking",
        "description_ar": "\u0645\u0641\u0627\u0635\u0644: \u0646\u0637\u0627\u0642 \u062d\u0631\u0643\u0629\u060c \u0637\u0642\u0637\u0642\u0629\u060c \u062a\u064a\u0628\u0633\u060c \u0623\u0644\u0645 \u0639\u0645\u064a\u0642\u060c \u0639\u062f\u0645 \u0627\u0633\u062a\u0642\u0631\u0627\u0631\u060c \u0627\u0646\u062c\u0645\u0627\u062f",
        "fallback_q_en": "How does the joint feel — is it stiff, swollen, clicking, or giving way? Is the range of motion reduced compared to the other side?",
        "fallback_q_ar": "\u0643\u064a\u0641 \u064a\u0628\u062f\u0648 \u0627\u0644\u0645\u0641\u0635\u0644 \u2014 \u0647\u0644 \u0647\u0648 \u0645\u062a\u064a\u0628\u0633\u060c \u0645\u062a\u0648\u0631\u0645\u060c \u064a\u0635\u062f\u0631 \u0637\u0642\u0637\u0642\u0629\u060c \u0623\u0648 \u064a\u0641\u0642\u062f \u062b\u0628\u0627\u062a\u0647 \u0641\u062c\u0623\u0629\u061f \u0647\u0644 \u0645\u062f\u0627\u0647 \u0627\u0644\u062d\u0631\u0643\u064a \u0623\u0642\u0644 \u0645\u0646 \u0627\u0644\u062c\u0627\u0646\u0628 \u0627\u0644\u0622\u062e\u0631\u061f",
        "positive_keywords": {
            "en": ["stiff","click","pop","unstable","lock","grind","deep ache","swollen joint","limited range","joint pain","joint","give way","wobbly"],
            "ar": ["\u062a\u064a\u0628\u0633","\u0637\u0642\u0637\u0642\u0629","\u0639\u062f\u0645 \u0627\u0633\u062a\u0642\u0631\u0627\u0631","\u0645\u062d\u062f\u0648\u062f\u064a\u0629","\u0623\u0644\u0645 \u0627\u0644\u0645\u0641\u0635\u0644","\u0645\u0641\u0635\u0644","\u0627\u0646\u062c\u0645\u0627\u062f","\u062a\u0648\u0631\u0645 \u0627\u0644\u0645\u0641\u0635\u0644"],
        },
    },
    Layer.BONE: {
        "description_en": "bone: point tenderness, deformity, trauma history, suspected fracture, inability to bear weight",
        "description_ar": "\u0639\u0638\u0627\u0645: \u0623\u0644\u0645 \u0646\u0642\u0637\u064a\u060c \u062a\u0634\u0648\u0647\u060c \u062a\u0627\u0631\u064a\u062e \u0635\u062f\u0645\u0629\u060c \u0627\u0634\u062a\u0628\u0627\u0647 \u0628\u0643\u0633\u0631\u060c \u0639\u062f\u0645 \u062a\u062d\u0645\u0644 \u0627\u0644\u0648\u0632\u0646",
        "fallback_q_en": "Did you have a fall, direct blow, or sudden injury? Is there a spot on the bone that hurts sharply when pressed — more than the surrounding area?",
        "fallback_q_ar": "\u0647\u0644 \u062a\u0639\u0631\u0636\u062a \u0644\u0633\u0642\u0648\u0637 \u0623\u0648 \u0636\u0631\u0628\u0629 \u0645\u0628\u0627\u0634\u0631\u0629 \u0623\u0648 \u0625\u0635\u0627\u0628\u0629 \u0645\u0641\u0627\u062c\u0626\u0629\u061f \u0647\u0644 \u062a\u0648\u062c\u062f \u0646\u0642\u0637\u0629 \u0628\u0639\u064a\u0646\u0647\u0627 \u0639\u0644\u0649 \u0627\u0644\u0639\u0638\u0645 \u062a\u0624\u0644\u0645 \u0628\u0634\u062f\u0629 \u0639\u0646\u062f \u0627\u0644\u0636\u063a\u0637 \u0639\u0644\u064a\u0647\u0627\u061f",
        "positive_keywords": {
            "en": ["trauma","injury","fell","fall","hit","fracture","crack","break","deform","point tender","bone pain","broken","snap","cannot walk","cannot bear"],
            "ar": ["\u0625\u0635\u0627\u0628\u0629","\u0633\u0642\u0648\u0637","\u0643\u0633\u0631","\u062a\u0634\u0648\u0647","\u0623\u0644\u0645 \u0646\u0642\u0637\u064a","\u0639\u0638\u0645\u0629","\u0635\u062f\u0645\u0629","\u0636\u0631\u0628\u0629","\u0644\u0627 \u0623\u0633\u062a\u0637\u064a\u0639 \u0627\u0644\u0645\u0634\u064a"],
        },
    },
    Layer.NERVE: {
        "description_en": "nerves: numbness, tingling, burning, electric sensation, radiating pain, dermatomal weakness",
        "description_ar": "\u0623\u0639\u0635\u0627\u0628: \u062a\u062e\u062f\u064a\u0631\u060c \u062a\u0646\u0645\u064a\u0644\u060c \u062d\u0631\u0642\u0629\u060c \u0625\u062d\u0633\u0627\u0633 \u0643\u0647\u0631\u0628\u0627\u0626\u064a\u060c \u0623\u0644\u0645 \u0645\u0646\u062a\u0634\u0631\u060c \u0636\u0639\u0641 \u062d\u0633\u064a \u0648\u062d\u0631\u0643\u064a",
        "fallback_q_en": "Do you experience any numbness, pins-and-needles, tingling, burning, or electric-shock sensation? Does the pain radiate down a limb or follow a specific path?",
        "fallback_q_ar": "\u0647\u0644 \u062a\u0634\u0639\u0631 \u0628\u062a\u062e\u062f\u064a\u0631 \u0623\u0648 \u062a\u0646\u0645\u064a\u0644 \u0623\u0648 \u062d\u0631\u0642\u0629 \u0623\u0648 \u0625\u062d\u0633\u0627\u0633 \u0643\u0627\u0644\u0643\u0647\u0631\u0628\u0627\u0621\u061f \u0647\u0644 \u064a\u0633\u064a\u0631 \u0627\u0644\u0623\u0644\u0645 \u0639\u0644\u0649 \u0637\u0648\u0644 \u0627\u0644\u0637\u0631\u0641 \u0623\u0648 \u064a\u062a\u0628\u0639 \u0645\u0633\u0627\u0631\u0627\u064b \u0645\u062d\u062f\u062f\u0627\u064b\u061f",
        "positive_keywords": {
            "en": ["numb","tingle","burn","radiat","shoot","electric","pins","needle","loss of feeling","sensation","nerve","shock","weak grip","drop foot"],
            "ar": ["\u062a\u062e\u062f\u064a\u0631","\u062a\u0646\u0645\u064a\u0644","\u062d\u0631\u0642\u0629","\u0627\u0646\u062a\u0634\u0627\u0631","\u0635\u0639\u0642","\u0641\u0642\u062f\u0627\u0646 \u0625\u062d\u0633\u0627\u0633","\u0643\u0647\u0631\u0628\u0627\u0621","\u0639\u0635\u0628","\u0625\u0628\u0631\u0629"],
        },
    },
    Layer.VASCULAR: {
        "description_en": "blood vessels: coldness, pallor, cyanosis, throbbing pain, claudication, poor capillary refill",
        "description_ar": "\u0623\u0648\u0639\u064a\u0629 \u062f\u0645\u0648\u064a\u0629: \u0628\u0631\u0648\u062f\u0629\u060c \u0634\u062d\u0648\u0628\u060c \u0632\u0631\u0642\u0629\u060c \u0623\u0644\u0645 \u0646\u0627\u0628\u0636\u060c \u062a\u0639\u0628 \u0648\u0639\u0627\u0626\u064a\u060c \u0636\u0639\u0641 \u0627\u0644\u0634\u0639\u064a\u0631\u0627\u062a",
        "fallback_q_en": "Is the area noticeably cold, pale, or slightly blue compared to the other side? Do you feel a throbbing or pulsating sensation, or pain that comes on with walking?",
        "fallback_q_ar": "\u0647\u0644 \u0627\u0644\u0645\u0646\u0637\u0642\u0629 \u0628\u0627\u0631\u062f\u0629 \u0623\u0648 \u0634\u0627\u062d\u0628\u0629 \u0623\u0648 \u0645\u0632\u0631\u0642\u0629 \u0628\u0634\u0643\u0644 \u0648\u0627\u0636\u062d \u0645\u0642\u0627\u0631\u0646\u0629 \u0628\u0627\u0644\u062c\u0627\u0646\u0628 \u0627\u0644\u0622\u062e\u0631\u061f \u0647\u0644 \u062a\u0634\u0639\u0631 \u0628\u0646\u0628\u0636\u0627\u0646 \u0623\u0648 \u0623\u0644\u0645 \u064a\u0623\u062a\u064a \u0639\u0646\u062f \u0627\u0644\u0645\u0634\u064a\u061f",
        "positive_keywords": {
            "en": ["cold","pale","blue","cyanosis","throb","puls","color change","discolor","capillary","veins","claudication","cramp walking"],
            "ar": ["\u0628\u0627\u0631\u062f","\u0634\u0627\u062d\u0628","\u0627\u0632\u0631\u0642\u0627\u0642","\u0646\u0627\u0628\u0636","\u062a\u063a\u064a\u0631 \u0644\u0648\u0646","\u0623\u0648\u0631\u062f\u0629","\u062a\u0639\u0628 \u0639\u0646\u062f \u0627\u0644\u0645\u0634\u064a"],
        },
    },
}

BODY_PART_HINTS: Dict[str, Dict[str, str]] = {
    "hand":       {"en":"Consider finger flexion, grip strength, intrinsic muscle wasting, and median/ulnar/radial nerve territories.",
                   "ar":"\u062e\u0630 \u0628\u0639\u064a\u0646 \u0627\u0644\u0627\u0639\u062a\u0628\u0627\u0631 \u062b\u0646\u064a \u0627\u0644\u0623\u0635\u0627\u0628\u0639 \u0648\u0642\u0648\u0629 \u0627\u0644\u0642\u0628\u0636\u0629 \u0648\u0636\u0645\u0648\u0631 \u0639\u0636\u0644\u0627\u062a \u0627\u0644\u064a\u062f \u0648\u0645\u0646\u0627\u0637\u0642 \u0627\u0644\u0623\u0639\u0635\u0627\u0628."},
    "knee":       {"en":"Consider meniscal signs (McMurray), ACL/PCL laxity, patellar tracking, and popliteal vascular status.",
                   "ar":"\u062e\u0630 \u0628\u0639\u064a\u0646 \u0627\u0644\u0627\u0639\u062a\u0628\u0627\u0631 \u0639\u0644\u0627\u0645\u0627\u062a \u0627\u0644\u063a\u0636\u0631\u0648\u0641 \u0627\u0644\u0647\u0644\u0627\u0644\u064a\u0629 \u0648\u0627\u0644\u0631\u0628\u0627\u0637 \u0627\u0644\u0635\u0644\u064a\u0628\u064a \u0648\u062a\u062a\u0628\u0639 \u0627\u0644\u0631\u0636\u0641\u0629."},
    "lower_back": {"en":"Focus on radiation to lower limb (L4/L5/S1 dermatomes), SLR test, and neurological deficits.",
                   "ar":"\u0631\u0643\u0632 \u0639\u0644\u0649 \u0627\u0646\u062a\u0634\u0627\u0631 \u0627\u0644\u0623\u0644\u0645 \u0644\u0644\u0637\u0631\u0641 \u0627\u0644\u0633\u0641\u0644\u064a \u0648\u0627\u0644\u0639\u062c\u0632 \u0627\u0644\u0639\u0635\u0628\u064a."},
    "shoulder":   {"en":"Consider rotator cuff integrity, AC joint, biceps tendon, and brachial plexus involvement.",
                   "ar":"\u062e\u0630 \u0628\u0639\u064a\u0646 \u0627\u0644\u0627\u0639\u062a\u0628\u0627\u0631 \u0633\u0644\u0627\u0645\u0629 \u0627\u0644\u0643\u0641\u0629 \u0627\u0644\u0645\u062f\u0648\u0631\u0629 \u0648\u0645\u0641\u0635\u0644 \u0627\u0644\u0623\u062e\u0631\u0645\u064a \u0627\u0644\u062a\u0631\u0642\u0648\u062a\u064a."},
    "foot":       {"en":"Consider plantar fascia, Achilles tendon, tarsal tunnel syndrome, and peripheral vascular disease.",
                   "ar":"\u062e\u0630 \u0628\u0639\u064a\u0646 \u0627\u0644\u0627\u0639\u062a\u0628\u0627\u0631 \u0627\u0644\u0644\u0641\u0627\u0641\u0629 \u0627\u0644\u062e\u0645\u0633\u064a\u0629 \u0648\u0648\u062a\u0631 \u0622\u0634\u064a\u0644 \u0648\u0645\u062a\u0644\u0627\u0632\u0645\u0629 \u0627\u0644\u0646\u0641\u0642 \u0627\u0644\u0631\u0633\u063a\u064a."},
}

def get_body_part_hint(body_part: str) -> Dict[str, str]:
    return BODY_PART_HINTS.get(body_part.lower(), {"en":"","ar":""})

@dataclass
class LayerState:
    layer:             Layer
    confidence:        float = 0.0
    symptom_positive:  bool  = False
    symptom_uncertain: bool  = False
    symptom_detail:    str   = ""
    followups:         int   = 0

class SessionState:
    def __init__(self, session_id: str, body_part: str):
        self.session_id   = session_id
        self.body_part    = body_part
        self.layer_index  = 0
        self.layers: List[LayerState] = [LayerState(layer=l) for l in LAYER_ORDER]
        self.history: List[Dict] = []
        self.is_complete  = False
        self.preferred_lang      = "en"
        self.session_status      = "ACTIVE"
        self.previous_layer: Optional[str] = None
        self.remove_previous_layer: bool   = False
        self.force_llm_question: bool      = False
        self.last_keyword_matches: int     = 0

    @property
    def current_layer(self) -> Layer:
        return LAYER_ORDER[self.layer_index] if self.layer_index < len(LAYER_ORDER) else LAYER_ORDER[-1]

    @property
    def current_layer_state(self) -> LayerState:
        return self.layers[self.layer_index]

print("Layer definitions and session state loaded.")

## Cell 4 — Language Detection & State Machine
- Early Exit
- Smart Jump
- SOCRATES follow-up

In [ ]:
# Cell 4 — Language Detection & State Machine

def detect_language(text: str) -> str:
    return "ar" if re.findall(r'[\u0600-\u06FF]', text) else "en"

class AgentAction(Enum):
    ASK_QUESTION    = "ask_question"
    ADVANCE_LAYER   = "advance_layer"
    GENERATE_REPORT = "generate_report"
    EMERGENCY       = "emergency"

class LayerStateMachine:
    def __init__(self):
        self.neg_en = r"\b(no|not|none|never|don't|doesn't|didn't|isn't|aren't|wasn't|weren't|nothing|without)\b"
        self.neg_ar = r"\b(\u0644\u0627|\u0644\u064a\u0633|\u0645\u0627|\u0644\u0645|\u0644\u0646|\u063a\u064a\u0631|\u0628\u062f\u0648\u0646|\u0645\u0641\u064a\u0634|\u0644\u0627 \u064a\u0648\u062c\u062f)\b"
        self.unc_en = r"\b(maybe|perhaps|not sure|possibly|sometimes|a little|slightly|kind of|sort of|i think)\b"
        self.unc_ar = r"\b(\u0645\u0645\u0643\u0646|\u0631\u0628\u0645\u0627|\u0645\u0634 \u0645\u062a\u0623\u0643\u062f|\u0645\u0634 \u0639\u0627\u0631\u0641|\u0623\u062d\u064a\u0627\u0646\u0627\u064b|\u0634\u0648\u064a\u0629|\u0646\u0648\u0639\u0627\u064b \u0645\u0627|\u0645\u0634 \u0645\u062a\u0623\u0643\u062f)\b"

    # ── Helpers ────────────────────────────────────────────────────────────────

    def _check_emergency(self, text: str) -> bool:
        t = text.lower()
        return any(kw in t for lang in ["en", "ar"] for kw in EMERGENCY_KEYWORDS[lang])

    def _score_layer(self, text: str, layer: Layer) -> int:
        """Raw keyword count for a layer — used for smart jump and entry detection."""
        t = text.lower()
        return sum(
            1 for lang in ["en", "ar"]
            for kw in LAYER_META[layer]["positive_keywords"][lang]
            if kw in t
        )

    def _detect_symptom(self, text: str, layer: Layer) -> Tuple[float, bool, bool, int]:
        """
        Returns (confidence, is_positive, is_uncertain, keyword_matches).
        Negation check happens first — if patient denies, always returns zero.
        """
        t = text.lower()

        has_negation = bool(re.search(self.neg_en, t) or re.search(self.neg_ar, t))
        if has_negation:
            return 0.0, False, False, 0

        is_uncertain = bool(re.search(self.unc_en, t) or re.search(self.unc_ar, t))
        matches      = self._score_layer(t, layer)

        if matches == 0:
            return (0.1, False, True, 0) if is_uncertain else (0.0, False, False, 0)

        confidence = min(0.4 + matches * 0.2, 1.0)
        if is_uncertain:
            confidence *= 0.6

        return confidence, confidence >= LOW_CONFIDENCE, is_uncertain, matches

    # ── Early Exit ─────────────────────────────────────────────────────────────

    def check_early_exit(self, state: "SessionState") -> bool:
        """
        Auto-generates report (without waiting for all layers) when evidence
        is strong enough — exactly what the doctor requested.

        Rule A — One layer is highly confident (>= EARLY_EXIT_CONFIDENCE).
        Rule B — Multiple positive layers with solid average confidence.
        """
        positive = [ls for ls in state.layers if ls.symptom_positive]
        if not positive:
            return False

        # Rule A
        if any(ls.confidence >= EARLY_EXIT_CONFIDENCE for ls in positive):
            print(f"[Early Exit] Rule A — high confidence in one layer. Auto-report.")
            return True

        # Rule B
        if len(positive) >= MIN_LAYERS_AUTO_EXIT:
            avg = sum(ls.confidence for ls in positive) / len(positive)
            if avg >= MULTI_LAYER_AVG_CONFIDENCE:
                print(f"[Early Exit] Rule B — {len(positive)} positive layers, avg {avg:.0%}. Auto-report.")
                return True

        return False

    # ── Smart Jump ─────────────────────────────────────────────────────────────

    def try_smart_jump(self, state: "SessionState", text: str) -> bool:
        """
        If patient's message matches a completely different layer far better
        than the current one, jump there directly — skip irrelevant layers.

        Example: current=SKIN, patient says "I fell and heard a crack"
        → BONE scores 3, SKIN scores 0 → jump to BONE.
        """
        curr_score = self._score_layer(text, state.current_layer)
        best_i     = state.layer_index
        best_score = curr_score

        for i, layer in enumerate(LAYER_ORDER):
            if i == state.layer_index:
                continue
            s = self._score_layer(text, layer)
            if s > best_score:
                best_i, best_score = i, s

        jumped = (
            best_i != state.layer_index
            and best_score >= SMART_JUMP_MIN_SCORE
            and best_score > curr_score + SMART_JUMP_MARGIN
        )

        if jumped:
            print(f"[Smart Jump] {state.current_layer.value} -> {LAYER_ORDER[best_i].value} (score={best_score})")
            state.previous_layer        = state.current_layer.value
            state.remove_previous_layer = True
            state.layer_index           = best_i
            state.force_llm_question    = False   # fresh layer → use fallback_q

        return jumped

    # ── Entry Detection ────────────────────────────────────────────────────────

    def detect_best_layer(self, text: str) -> int:
        """Used in /start to pick the right entry layer from chief_complaint. Zero GPU."""
        scores = {i: self._score_layer(text, l) for i, l in enumerate(LAYER_ORDER)}
        best   = max(scores, key=scores.get)
        return best if scores[best] > 0 else 0

    # ── Severity ───────────────────────────────────────────────────────────────

    def compute_severity(self, layers: List[LayerState]) -> Tuple[str, List[str]]:
        """Computes final severity score and correlation warnings across all layers."""
        score    = 0.0
        positive = set()

        for ls in layers:
            if ls.symptom_positive:
                score += LAYER_SEVERITY_WEIGHT[ls.layer.value] * ls.confidence
                positive.add(ls.layer.value)

        warnings = []
        for required, label, boost in CORRELATION_RULES:
            if required.issubset(positive):
                score += boost
                warnings.append(label)

        if score > 2.5:   sev = "CRITICAL"
        elif score > 1.5: sev = "HIGH"
        elif score > 0.8: sev = "MODERATE"
        else:             sev = "LOW"

        return sev, warnings

    # ── Decision Logic ─────────────────────────────────────────────────────────

    def _advance_or_report(self, state: "SessionState") -> AgentAction:
        if state.layer_index < len(LAYER_ORDER) - 1:
            return AgentAction.ADVANCE_LAYER
        state.session_status = "COMPLETED"
        return AgentAction.GENERATE_REPORT

    def decide(self, state: "SessionState", user_message: str) -> AgentAction:
        # Reset per-turn flags
        state.remove_previous_layer = False
        state.force_llm_question    = False
        state.last_keyword_matches  = 0

        # 1. Emergency
        if self._check_emergency(user_message):
            state.session_status = "EMERGENCY"
            return AgentAction.EMERGENCY

        # 2. Smart Jump — patient's words clearly belong to another layer
        if self.try_smart_jump(state, user_message):
            if self.check_early_exit(state):
                state.session_status = "COMPLETED"
                return AgentAction.GENERATE_REPORT
            return AgentAction.ASK_QUESTION

        # 3. Detect symptom in current layer
        ls   = state.current_layer_state
        conf, pos, unc, matches = self._detect_symptom(user_message, state.current_layer)
        state.last_keyword_matches = matches

        # Update layer state only if new evidence is stronger
        if conf > ls.confidence:
            ls.confidence        = conf
            ls.symptom_positive  = pos
            ls.symptom_uncertain = unc
            ls.symptom_detail    = user_message

        # 4. Early exit after every update
        if self.check_early_exit(state):
            state.session_status = "COMPLETED"
            return AgentAction.GENERATE_REPORT

        # 5. Positive symptom — ask LLM follow-up (SOCRATES style)
        if ls.symptom_positive:
            if ls.followups < MAX_FOLLOWUPS_WITH_SYMPTOM:
                ls.followups                += 1
                state.force_llm_question     = True
                return AgentAction.ASK_QUESTION
            return self._advance_or_report(state)

        # 6. Uncertain — one LLM clarifying question
        if unc and ls.followups == 0:
            ls.followups                += 1
            state.force_llm_question     = True
            return AgentAction.ASK_QUESTION

        # 7. No keyword match at all — let LLM ask smart question (semantic path)
        #    Agent will run semantic_detect before generating the question
        if matches == 0 and ls.followups == 0:
            ls.followups                += 1
            state.force_llm_question     = True
            return AgentAction.ASK_QUESTION

        # 8. Clear negative or exhausted attempts — advance
        return self._advance_or_report(state)

print("State machine loaded.")

## Cell 5 — LLMGenerator *(Fixes 1, 2 & 3 integrated)*

| Fix | Method | What it does |
|-----|--------|--------------|
| **Fix 1** | `_structured_generate()` | Delimiter-based extraction `<<<JSON>>>...<<<END>>>`. 3-layer fallback: delimiter → last JSON block → key-by-key regex. Replaces the brittle "Output ONLY JSON" approach. |
| **Fix 2** | `_context_aware_fallback()` | Scans last 4 patient messages for layer keywords before returning a pre-written question. Uses a targeted SOCRATES sub-question if keywords already mentioned — zero GPU cost. |
| **Fix 3** | `_summarize_confirmed_findings()` | Lightweight LLM call producing a 2-3 sentence summary of confirmed findings, injected into every prompt as "do NOT re-ask these". Cached per `(session_id, history_len)`. |

In [ ]:
# Cell 5 — LLMGenerator with Fixes 1, 2 & 3 integrated
#
# Fix 1: _structured_generate()          — delimiter-based JSON extraction (3-layer fallback)
# Fix 2: _context_aware_fallback()       — history-aware fallback question selection
# Fix 3: _summarize_confirmed_findings() — cached confirmed-findings injection into prompts

import re as _re
import json as _json
from typing import Optional, Tuple, List, Dict

_DELIM_OPEN   = "<<<JSON>>>"
_DELIM_CLOSE  = "<<<END>>>"
_SUMMARY_CACHE: dict = {}   # (session_id, history_len) -> summary str


class LLMGenerator:
    def __init__(self, model_id: str):
        global global_tokenizer, global_model
        self.tokenizer = global_tokenizer
        self.model     = global_model
        print("LLMGenerator linked to global GPU model.")

    # ── Core generation ────────────────────────────────────────────────────────

    def _raw_generate(self, prompt: str, max_new_tokens: int = 400) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens     = max_new_tokens,
                do_sample          = False,
                temperature        = 1.0,
                repetition_penalty = 1.15,
                pad_token_id       = self.tokenizer.eos_token_id,
            )
        new_tokens = out[0][inputs["input_ids"].shape[-1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    # ── Fix 1: Delimiter-based structured generation ───────────────────────────

    def _structured_generate(self, prompt: str, json_keys: list,
                              max_new_tokens: int = 300) -> Optional[dict]:
        """
        Replaces the brittle Output-ONLY-JSON approach.
        Asks the model to wrap its JSON in <<<JSON>>>...<<<END>>> delimiters,
        then tries 3 extraction methods in order:
          1. Delimiter capture  (primary)
          2. Last JSON block in raw text
          3. Key-by-key regex harvest
        """
        delim_instr = (
            "\n\nAfter your reasoning (if any), output the result between these exact delimiters:\n"
            f"{_DELIM_OPEN}{{...}}{_DELIM_CLOSE}\n"
            "No markdown, no backticks inside the delimiters."
        )
        augmented = (
            prompt.replace("[/INST]", delim_instr + "\n[/INST]", 1)
            if "[/INST]" in prompt else prompt + delim_instr
        )
        raw = self._raw_generate(augmented, max_new_tokens=max_new_tokens)

        # Attempt 1: delimiters
        m = _re.search(
            _re.escape(_DELIM_OPEN) + r"\s*(\{.*?\})\s*" + _re.escape(_DELIM_CLOSE),
            raw, _re.DOTALL
        )
        if m:
            try:
                parsed = _json.loads(m.group(1))
                if all(k in parsed for k in json_keys):
                    return parsed
            except _json.JSONDecodeError:
                pass

        # Attempt 2: last JSON block in text
        blocks = _re.findall(r"\{[^{}]*\}", raw, _re.DOTALL)
        for block in reversed(blocks):
            clean = _re.sub(r"```[a-z]*", "", block).strip()
            try:
                parsed = _json.loads(clean)
                if all(k in parsed for k in json_keys):
                    print("[OutputExtract] Delimiter missing — JSON recovered from text.")
                    return parsed
            except _json.JSONDecodeError:
                continue

        # Attempt 3: key-by-key regex harvest
        harvested = {}
        for key in json_keys:
            hit = _re.search(rf'"{_re.escape(key)}"\s*:\s*"([^"]+)"', raw)
            if hit:
                harvested[key] = hit.group(1).strip()
        if len(harvested) == len(json_keys):
            print("[OutputExtract] Recovered via key-by-key harvest.")
            return harvested

        print(f"[OutputExtract] WARNING: All extraction attempts failed. Raw (200): {raw[:200]}")
        return None

    # ── Helpers ────────────────────────────────────────────────────────────────

    def _extract_json(self, text: str) -> Optional[dict]:
        """Legacy helper kept for generate_report compatibility."""
        m = _re.search(r"\{[^{}]*\}", text, _re.DOTALL)
        if m:
            try: return _json.loads(m.group(0))
            except _json.JSONDecodeError: pass
        try: return _json.loads(text.strip())
        except _json.JSONDecodeError: return None

    def _last_patient_msg(self, history: List[Dict]) -> str:
        for m in reversed(history):
            if m.get("role") == "patient":
                return m.get("content_en", m.get("content", "")).strip()
        return ""

    def _history_str(self, history: List[Dict], n: int = 6) -> str:
        return "\n".join(
            f"{m['role'].capitalize()}: {m.get('content_en', m.get('content',''))}"
            for m in history[-n:]
        )

    # ── Fix 2: Context-aware fallback ─────────────────────────────────────────

    def _context_aware_fallback(self, layer, history: List[Dict],
                                preferred_lang: str) -> Tuple[str, str]:
        """
        Before returning the pre-written fallback question, scans the last 4
        patient messages. If they already mentioned a keyword for this layer,
        returns a targeted SOCRATES sub-question — zero GPU cost.
        """
        meta = LAYER_META[layer]
        recent_patient = " ".join(
            m.get("content_en", m.get("content", "")).lower()
            for m in history[-4:]
            if m.get("role") == "patient"
        )
        if not recent_patient:
            return meta["fallback_q_en"], meta["fallback_q_ar"]

        already_mentioned = any(
            kw in recent_patient
            for lang in ["en", "ar"]
            for kw in meta["positive_keywords"][lang]
        )
        if not already_mentioned:
            return meta["fallback_q_en"], meta["fallback_q_ar"]

        targeted = {
            "skin": {
                "en": "You mentioned a skin change — can you describe how it started and whether it has spread or changed in size?",
                "ar": "\u0630\u0643\u0631\u062a \u062a\u063a\u064a\u0631\u0627\u064b \u0641\u064a \u0627\u0644\u062c\u0644\u062f \u2014 \u0643\u064a\u0641 \u0628\u062f\u0623 \u0648\u0647\u0644 \u0627\u0646\u062a\u0634\u0631 \u0623\u0648 \u062a\u063a\u064a\u0631 \u062d\u062c\u0645\u0647\u061f",
            },
            "soft_tissue": {
                "en": "Since you mentioned tenderness or swelling — has it changed in size since it started, and does pressing on it change the pain?",
                "ar": "\u0628\u0645\u0627 \u0623\u0646\u0643 \u0630\u0643\u0631\u062a \u0623\u0644\u0645\u0627\u064b \u0623\u0648 \u062a\u0648\u0631\u0645\u0627\u064b \u2014 \u0647\u0644 \u062a\u063a\u064a\u0631 \u062d\u062c\u0645\u0647 \u0645\u0646\u0630 \u0627\u0644\u0628\u062f\u0627\u064a\u0629\u060c \u0648\u0647\u0644 \u0627\u0644\u0636\u063a\u0637 \u0639\u0644\u064a\u0647 \u064a\u063a\u064a\u0631 \u0627\u0644\u0623\u0644\u0645\u061f",
            },
            "muscle_tendon": {
                "en": "Given the muscle symptoms you described — on a scale of 0-10, how severe is the pain, and does rest improve it?",
                "ar": "\u0628\u0627\u0644\u0646\u0633\u0628\u0629 \u0644\u0623\u0639\u0631\u0627\u0636 \u0627\u0644\u0639\u0636\u0644\u0629 \u2014 \u0639\u0644\u0649 \u0645\u0642\u064a\u0627\u0633 0-10\u060c \u0645\u0627 \u0634\u062f\u0629 \u0627\u0644\u0623\u0644\u0645\u060c \u0648\u0647\u0644 \u0627\u0644\u0631\u0627\u062d\u0629 \u062a\u062d\u0633\u0651\u0646\u0647\u061f",
            },
            "joint": {
                "en": "You mentioned joint symptoms — is the stiffness or pain worse in the morning, after rest, or with activity?",
                "ar": "\u0630\u0643\u0631\u062a \u0623\u0639\u0631\u0627\u0636\u0627\u064b \u0641\u064a \u0627\u0644\u0645\u0641\u0635\u0644 \u2014 \u0647\u0644 \u0627\u0644\u062a\u064a\u0628\u0633 \u0623\u0648 \u0627\u0644\u0623\u0644\u0645 \u0623\u0633\u0648\u0623 \u0641\u064a \u0627\u0644\u0635\u0628\u0627\u062d \u0623\u0648 \u0628\u0639\u062f \u0627\u0644\u0631\u0627\u062d\u0629 \u0623\u0648 \u0645\u0639 \u0627\u0644\u062d\u0631\u0643\u0629\u061f",
            },
            "bone": {
                "en": "Given what you described — can you point to the exact spot that hurts most when pressed, and can you bear weight on it?",
                "ar": "\u0628\u0646\u0627\u0621\u064b \u0639\u0644\u0649 \u0645\u0627 \u0648\u0635\u0641\u062a\u0647 \u2014 \u0647\u0644 \u064a\u0645\u0643\u0646\u0643 \u062a\u062d\u062f\u064a\u062f \u0627\u0644\u0646\u0642\u0637\u0629 \u0627\u0644\u062a\u064a \u062a\u0624\u0644\u0645 \u0623\u0643\u062b\u0631 \u0639\u0646\u062f \u0627\u0644\u0636\u063a\u0637\u060c \u0648\u0647\u0644 \u062a\u0633\u062a\u0637\u064a\u0639 \u062a\u062d\u0645\u0644 \u0648\u0632\u0646\u0643 \u0639\u0644\u064a\u0647\u0627\u061f",
            },
            "nerve": {
                "en": "Since you mentioned nerve-like symptoms — does the sensation travel down a specific path, and does any position make it better or worse?",
                "ar": "\u0628\u0645\u0627 \u0623\u0646\u0643 \u0630\u0643\u0631\u062a \u0623\u0639\u0631\u0627\u0636\u0627\u064b \u0639\u0635\u0628\u064a\u0629 \u2014 \u0647\u0644 \u0627\u0644\u0625\u062d\u0633\u0627\u0633 \u064a\u0633\u064a\u0631 \u0641\u064a \u0645\u0633\u0627\u0631 \u0645\u062d\u062f\u062f\u060c \u0648\u0647\u0644 \u0648\u0636\u0639\u064a\u0629 \u0645\u0639\u064a\u0646\u0629 \u062a\u062d\u0633\u0651\u0646\u0647 \u0623\u0648 \u062a\u0632\u064a\u062f\u0647\u061f",
            },
            "vascular": {
                "en": "Given the circulation symptoms you mentioned — does the coldness or color change affect the whole area or just the tips, and does elevating it help?",
                "ar": "\u0628\u0627\u0644\u0646\u0633\u0628\u0629 \u0644\u0623\u0639\u0631\u0627\u0636 \u0627\u0644\u062f\u0648\u0631\u0629 \u0627\u0644\u062f\u0645\u0648\u064a\u0629 \u2014 \u0647\u0644 \u0627\u0644\u0628\u0631\u0648\u062f\u0629 \u0623\u0648 \u062a\u063a\u064a\u0631 \u0627\u0644\u0644\u0648\u0646 \u064a\u0634\u0645\u0644 \u0627\u0644\u0645\u0646\u0637\u0642\u0629 \u0643\u0644\u0647\u0627 \u0623\u0645 \u0641\u0642\u0637 \u0627\u0644\u0623\u0637\u0631\u0627\u0641\u060c \u0648\u0647\u0644 \u0627\u0644\u0631\u0641\u0639 \u064a\u0633\u0627\u0639\u062f\u061f",
            },
        }
        t = targeted.get(layer.value)
        if t:
            print(f"[Fix2] Patient already mentioned {layer.value} keywords — targeted sub-question used.")
            return t["en"], t["ar"]
        return meta["fallback_q_en"], meta["fallback_q_ar"]

    # ── Fix 3: History summarization ───────────────────────────────────────────

    def _summarize_confirmed_findings(self, session_id: str,
                                      history: List[Dict], body_part: str) -> str:
        """
        Lightweight LLM call (~120 tokens) summarising confirmed findings.
        Injected into every subsequent prompt as do-NOT-re-ask context.
        Cached per (session_id, history_len).
        """
        if len(history) < 4:
            return ""
        cache_key = (session_id, len(history))
        if cache_key in _SUMMARY_CACHE:
            return _SUMMARY_CACHE[cache_key]

        history_str = "\n".join(
            f"{m['role'].capitalize()}: {m.get('content_en', m.get('content',''))}"
            for m in history[-8:]
        )
        prompt = (
            f"[INST] <<SYS>>\n"
            f"You are a clinical notes assistant. Read this conversation about a patient's {body_part}.\n\n"
            f"Write 2-3 short sentences summarising ONLY what has been CONFIRMED so far "
            f"(symptoms present, symptoms absent, severity if stated). "
            f"Do NOT speculate. Do NOT add diagnoses. Use clinical shorthand.\n\n"
            f'Output JSON with key "summary" (string).\n'
            f"<</SYS>>\n{history_str}\n[/INST]"
        )
        parsed  = self._structured_generate(prompt, json_keys=["summary"], max_new_tokens=120)
        summary = parsed["summary"].strip() if parsed and "summary" in parsed else ""
        if summary:
            print(f"[Fix3] Summary ({len(history)} turns): {summary[:80]}...")
            _SUMMARY_CACHE[cache_key] = summary
        else:
            print("[Fix3] Summary generation failed — proceeding without.")
        return summary

    # ── Semantic Detection ─────────────────────────────────────────────────────

    def semantic_detect(self, patient_message: str, layer) -> float:
        meta   = LAYER_META[layer]
        prompt = (
            f"[INST] <<SYS>>\n"
            f"You are a clinical triage classifier.\n"
            f"Assess whether this patient message suggests symptoms of: {layer.value}.\n"
            f"Layer definition: {meta['description_en']}\n"
            f"Patient said: \"{patient_message}\"\n\n"
            f"confidence = 0.0 (completely unrelated) to 1.0 (clearly related).\n"
            f'Output JSON with key "confidence" (float).\n'
            f"<</SYS>>\n[/INST]"
        )
        parsed = self._structured_generate(prompt, json_keys=["confidence"], max_new_tokens=80)
        if parsed:
            try: return max(0.0, min(1.0, float(parsed["confidence"])))
            except (ValueError, TypeError): pass
        return 0.0

    # ── Question Generation (Fixes 1 + 2 + 3 active) ──────────────────────────

    def generate_question(
        self,
        body_part:        str,
        layer,
        history:          List[Dict],
        preferred_lang:   str  = "en",
        is_followup:      bool = False,
        is_clarification: bool = False,
        force_llm:        bool = False,
        session_id:       str  = "unknown",
    ) -> Tuple[str, str]:
        meta = LAYER_META[layer]

        # Fast path — Fix 2 context-aware fallback (zero GPU cost)
        if not force_llm and not is_followup and not is_clarification:
            return self._context_aware_fallback(layer, history, preferred_lang)

        # Fix 3 — inject confirmed findings summary into prompt
        summary = self._summarize_confirmed_findings(session_id, history, body_part)
        summary_block = (
            f"\n\nCONFIRMED FINDINGS SO FAR (do NOT re-ask these):\n{summary}"
            if summary else ""
        )

        hint        = get_body_part_hint(body_part)
        last_msg    = self._last_patient_msg(history)
        history_str = self._history_str(history, n=6)
        socrates    = LAYER_SOCRATES.get(layer.value, "severity, onset, character, and associated symptoms")

        if force_llm and not is_followup and not is_clarification:
            prompt = (
                f"[INST] <<SYS>>\n"
                f"You are an experienced bilingual clinician assessing a patient's {body_part}.{summary_block}\n\n"
                f"THE PATIENT'S EXACT WORDS: \"{last_msg}\"\n\n"
                f"Ask ONE precise clinical question directly following from their words.\n"
                f'Output JSON with keys "question_en" and "question_ar".\n'
                f"<</SYS>>\nConversation:\n{history_str}\n[/INST]"
            )
        elif is_followup:
            prompt = (
                f"[INST] <<SYS>>\n"
                f"You are an experienced bilingual clinician. "
                f"The patient confirmed a {layer.value} symptom in their {body_part}.{summary_block}\n\n"
                f"PATIENT'S LAST MESSAGE: \"{last_msg}\"\n"
                f"ANATOMICAL CONTEXT: {hint['en']}\n"
                f"EXPLORE (SOCRATES): {socrates}\n\n"
                f"Ask ONE specific SOCRATES follow-up question — not already answered above.\n"
                f'Output JSON with keys "question_en" and "question_ar".\n'
                f"<</SYS>>\nConversation:\n{history_str}\n[/INST]"
            )
        else:
            prompt = (
                f"[INST] <<SYS>>\n"
                f"You are a bilingual clinician.{summary_block}\n\n"
                f"PATIENT'S LAST MESSAGE: \"{last_msg}\"\n"
                f"LAYER: {meta['description_en']}\n\n"
                f"Ask ONE gentle clarifying question.\n"
                f'Output JSON with keys "question_en" and "question_ar".\n'
                f"<</SYS>>\nConversation:\n{history_str}\n[/INST]"
            )

        # Fix 1 — delimiter-based extraction
        parsed = self._structured_generate(
            prompt, json_keys=["question_en", "question_ar"], max_new_tokens=220
        )
        if parsed:
            q_en = str(parsed["question_en"]).strip()
            q_ar = str(parsed["question_ar"]).strip()
            if len(q_en) > 10 and len(q_ar) > 5:
                return q_en, q_ar

        # Safety net
        return self._context_aware_fallback(layer, history, preferred_lang)

    # ── Report Generation (unchanged from original) ────────────────────────────

    def generate_report(self, body_part, layers, history, severity, correlation_warnings):
        positive_layers = [ls for ls in layers if ls.symptom_positive]
        symptom_lines   = [
            f"- {ls.layer.value.replace('_',' ').title()} "
            f"(confidence {ls.confidence:.0%}): {ls.symptom_detail.strip()}"
            for ls in positive_layers
        ]
        symptom_summary  = "\n".join(symptom_lines) if symptom_lines else "No specific symptoms confirmed."
        correlation_text = (
            "CROSS-LAYER ALERTS:\n" + "\n".join(f"  - {w}" for w in correlation_warnings)
        ) if correlation_warnings else ""
        avg_conf   = (sum(ls.confidence for ls in positive_layers) / len(positive_layers)
                      if positive_layers else 0.0)
        conf_label = "HIGH" if avg_conf >= 0.7 else "MODERATE" if avg_conf >= 0.45 else "LOW"

        prompt = (
            f"[INST] <<SYS>>\n"
            f"You are a senior clinical triage AI producing a structured medical triage report.\n"
            f"Body region: {body_part}\n\n"
            f"CONFIRMED CLINICAL FINDINGS:\n{symptom_summary}\n"
            f"{correlation_text}\n\n"
            f"PREDETERMINED SEVERITY: {severity}  — use this EXACTLY, do not change it.\n"
            f"ASSESSMENT CONFIDENCE: {conf_label}\n\n"
            f"STRICT RULES:\n"
            f"- Conservative clinical language. This is TRIAGE, not a final diagnosis.\n"
            f"- Do not invent findings not listed above.\n"
            f"- differentials: 2-3 plausible diagnoses ordered by probability.\n"
            f"- red_flags: urgent/worrying features from findings (empty list if none).\n"
            f"- Output ONLY valid JSON. No markdown. No extra text.\n\n"
            f'JSON FORMAT:\n{{\n  "diagnosis_en":"...","diagnosis_ar":"...","severity":"{severity}",\n'
            f'  "assessment_confidence":"{conf_label}","specialist_en":"...","specialist_ar":"...",\n'
            f'  "immediate_actions_en":["..."],"immediate_actions_ar":["..."],\n'
            f'  "lifestyle_advice_en":["..."],"lifestyle_advice_ar":["..."],\n'
            f'  "differentials":[{{"diagnosis":"...","probability":"HIGH"}}],\n'
            f'  "red_flags":["..."],"follow_up_en":"...","follow_up_ar":"...",\n'
            f'  "summary_en":"...","summary_ar":"..."\n}}\n'
            f"<</SYS>>\n[/INST]"
        )
        raw    = self._raw_generate(prompt, max_new_tokens=900)
        parsed = self._extract_json(raw)
        required = [
            "diagnosis_en","diagnosis_ar","severity","specialist_en","specialist_ar",
            "immediate_actions_en","immediate_actions_ar","summary_en","summary_ar",
        ]
        if parsed and all(k in parsed for k in required):
            parsed["severity"]              = severity
            parsed["assessment_confidence"] = conf_label
            if correlation_warnings:
                parsed["correlation_alerts"] = correlation_warnings
            parsed.setdefault("differentials",       [])
            parsed.setdefault("red_flags",           [])
            parsed.setdefault("lifestyle_advice_en", [])
            parsed.setdefault("lifestyle_advice_ar", [])
            parsed.setdefault("follow_up_en", "Please consult a specialist as soon as possible.")
            parsed.setdefault("follow_up_ar", "\u064a\u0631\u062c\u0649 \u0645\u0631\u0627\u062c\u0639\u0629 \u0623\u062e\u0635\u0627\u0626\u064a \u0641\u064a \u0623\u0642\u0631\u0628 \u0648\u0642\u062a.")
            return parsed

        return {
            "diagnosis_en":          "Clinical impression pending specialist review.",
            "diagnosis_ar":          "\u0627\u0644\u0627\u0646\u0637\u0628\u0627\u0639 \u0627\u0644\u0633\u0631\u064a\u0631\u064a \u0641\u064a \u0627\u0646\u062a\u0638\u0627\u0631 \u0645\u0631\u0627\u062c\u0639\u0629 \u0627\u0644\u0623\u062e\u0635\u0627\u0626\u064a.",
            "severity":              severity,
            "assessment_confidence": conf_label,
            "specialist_en":         "General Practitioner",
            "specialist_ar":         "\u0637\u0628\u064a\u0628 \u0639\u0627\u0645",
            "immediate_actions_en":  ["Schedule an in-person consultation.", "Avoid strenuous activity."],
            "immediate_actions_ar":  ["\u062d\u062f\u062f \u0645\u0648\u0639\u062f\u0627\u064b \u0644\u0644\u0643\u0634\u0641.", "\u062a\u062c\u0646\u0628 \u0627\u0644\u0645\u062c\u0647\u0648\u062f \u0627\u0644\u0634\u062f\u064a\u062f."],
            "lifestyle_advice_en":   ["Rest the affected area.", "Apply ice 15 min/hour if swelling."],
            "lifestyle_advice_ar":   ["\u0623\u0631\u062d \u0627\u0644\u0645\u0646\u0637\u0642\u0629 \u0627\u0644\u0645\u062a\u0623\u0644\u0645\u0629.", "\u0636\u0639 \u062b\u0644\u062c\u0627\u064b 15 \u062f\u0642\u064a\u0642\u0629 \u0643\u0644 \u0633\u0627\u0639\u0629 \u0625\u0646 \u0648\u062c\u062f \u062a\u0648\u0631\u0645."],
            "differentials":         [],
            "red_flags":             [],
            "follow_up_en":          "Please consult a specialist as soon as possible.",
            "follow_up_ar":          "\u064a\u0631\u062c\u0649 \u0645\u0631\u0627\u062c\u0639\u0629 \u0623\u062e\u0635\u0627\u0626\u064a \u0641\u064a \u0623\u0642\u0631\u0628 \u0648\u0642\u062a.",
            "summary_en":            f"Patient reported symptoms in the {body_part}. Specialist evaluation recommended.",
            "summary_ar":            f"\u0623\u0641\u0627\u062f \u0627\u0644\u0645\u0631\u064a\u0636 \u0628\u0623\u0639\u0631\u0627\u0636 \u0641\u064a {body_part}. \u064a\u0648\u0635\u0649 \u0628\u062a\u0642\u064a\u064a\u0645 \u0645\u062a\u062e\u0635\u0635.",
            "correlation_alerts":    correlation_warnings,
        }


print("\u2705 Cell 5 \u2014 LLMGenerator with Fixes 1, 2 & 3 loaded.")


## Cell 6 — Medical Agent *(session_id forwarded for Fix 3)*

Only change from original: `session_id` is now passed to both `generate_question()` call sites so Fix 3's summary cache keys correctly per session.

In [ ]:
# Cell 6 — Medical Agent (session_id forwarded to generate_question for Fix 3)
# Cell 6 — Medical Agent

def _run_report_in_background(
    session_id: str, body_part: str,
    layers_snap: list, history_snap: list,
    severity: str, correlation_warnings: list,
    llm: "LLMGenerator",
):
    try:
        print(f"[Report] Generating for session {session_id}...")
        report = llm.generate_report(body_part, layers_snap, history_snap, severity, correlation_warnings)
        reports_db[session_id] = {"status": "COMPLETED", "report": report}
        save_report_to_disk(session_id, report)
        print(f"[Report] Done — session {session_id}.")
    except Exception as e:
        print(f"[Report] Error — session {session_id}: {e}")
        reports_db[session_id] = {"status": "FAILED", "error": str(e)}


class MedicalAgent:
    def __init__(self, llm: LLMGenerator):
        self.llm      = llm
        self.fsm      = LayerStateMachine()
        self.sessions: Dict[str, SessionState] = {}

    # ── Session Management ─────────────────────────────────────────────────────

    def _get_or_create(self, session_id: str, body_part: str) -> SessionState:
        if session_id not in self.sessions:
            self.sessions[session_id] = SessionState(session_id, body_part)
        return self.sessions[session_id]

    def cleanup_session(self, session_id: str):
        """
        Removes session from memory after report is confirmed delivered.
        Report file stays on disk permanently — if client reconnects
        with the same session_id they get the stored report, not new questions.
        """
        self.sessions.pop(session_id, None)
        reports_db.pop(session_id, None)
        print(f"[Cleanup] Session {session_id} removed from memory.")

    # ── Report Trigger ─────────────────────────────────────────────────────────

    def _trigger_report(self, state: SessionState) -> dict:
        """Marks session complete and launches background report thread."""
        state.is_complete    = True
        state.session_status = "COMPLETED"
        severity, corr       = self.fsm.compute_severity(state.layers)

        reports_db[state.session_id] = {"status": "PROCESSING"}

        threading.Thread(
            target=_run_report_in_background,
            args=(
                state.session_id, state.body_part,
                copy.deepcopy(state.layers),
                copy.deepcopy(state.history),
                severity, corr, self.llm,
            ),
            daemon=True,
        ).start()

        return {
            "status":         "PROCESSING",
            "session_status": "COMPLETED",
            "session_id":     state.session_id,
            "message":        "Report is being generated. Poll /get_report/{session_id} every 10 seconds.",
        }

    # ── /start ─────────────────────────────────────────────────────────────────

    def start_session(self, body_part: str, chief_complaint: Optional[str] = None) -> dict:
        session_id = str(uuid.uuid4())
        state      = self._get_or_create(session_id, body_part)

        if chief_complaint:
            state.preferred_lang = detect_language(chief_complaint)
            state.layer_index    = self.fsm.detect_best_layer(chief_complaint)
            state.history.append({
                "role":       "patient",
                "content_en": chief_complaint,
                "content_ar": chief_complaint,
                "lang":       state.preferred_lang,
            })
            if state.layer_index > 0:
                print(f"[Smart Entry] Starting at: {state.current_layer.value}")

        # First question always uses fallback_q — fast, no GPU
        q_en, q_ar = self.llm.generate_question(
            body_part=state.body_part, layer=state.current_layer,
            history=state.history, preferred_lang=state.preferred_lang,
            force_llm=False, session_id=session_id,
        )
        state.history.append({"role": "doctor", "content_en": q_en, "content_ar": q_ar})

        return {
            "session_id":        session_id,
            "first_question_en": q_en,
            "first_question_ar": q_ar,
            "layer":             state.current_layer.value,
            "layer_index":       state.layer_index,
        }

    # ── /chat ──────────────────────────────────────────────────────────────────

    def process(self, session_id: str, body_part: str, user_message: str) -> dict:
        state = self._get_or_create(session_id, body_part)

        # Already complete — return current report status or load from disk
        if state.is_complete:
            entry = reports_db.get(session_id)
            if entry:
                return entry
            saved = load_report_from_disk(session_id)
            if saved:
                return {"status": "COMPLETED", "report": saved}
            return {"status": "PROCESSING", "message": "Poll /get_report/{session_id}."}

        # Detect language on first real patient message
        if len(state.history) <= 1:
            state.preferred_lang = detect_language(user_message)

        state.history.append({
            "role":       "patient",
            "content_en": user_message,
            "content_ar": user_message,
            "lang":       detect_language(user_message),
        })

        # FSM decision
        action = self.fsm.decide(state, user_message)

        # ── Emergency ──────────────────────────────────────────────────────────
        if action == AgentAction.EMERGENCY:
            state.is_complete    = True
            state.session_status = "EMERGENCY"
            return {
                "status":         "EMERGENCY",
                "session_status": "EMERGENCY",
                "message_en":     "Your symptoms may indicate a life-threatening emergency. "
                                  "Call emergency services (123) or go to the nearest ER immediately.",
                "message_ar":     "\u0642\u062f \u062a\u0634\u064a\u0631 \u0623\u0639\u0631\u0627\u0636\u0643 \u0625\u0644\u0649 \u062d\u0627\u0644\u0629 \u0637\u0627\u0631\u0626\u0629. "
                                  "\u0627\u062a\u0635\u0644 \u0628\u0627\u0644\u0637\u0648\u0627\u0631\u0626 (123) \u0623\u0648 "
                                  "\u0627\u0630\u0647\u0628 \u0641\u0648\u0631\u0627\u064b \u0644\u0623\u0642\u0631\u0628 \u063a\u0631\u0641\u0629 \u0637\u0648\u0627\u0631\u0626.",
            }

        # ── Auto-Report (Early Exit triggered by FSM) ──────────────────────────
        if action == AgentAction.GENERATE_REPORT:
            return self._trigger_report(state)

        # ── Advance Layer ──────────────────────────────────────────────────────
        if action == AgentAction.ADVANCE_LAYER:
            state.previous_layer        = state.current_layer.value
            state.remove_previous_layer = True
            state.layer_index          += 1
            state.force_llm_question    = False
            action = AgentAction.ASK_QUESTION

        # ── Semantic Detection ─────────────────────────────────────────────────
        # Runs ONLY when keywords found nothing — uses LLM to check if the
        # patient's message semantically matches the current layer.
        # This bridges the gap between keywords and full understanding.
        if (action == AgentAction.ASK_QUESTION
                and state.last_keyword_matches == 0
                and state.force_llm_question):

            last_msg = state.history[-1].get("content_en", "")
            if last_msg:
                sem_conf = self.llm.semantic_detect(last_msg, state.current_layer)
                print(f"[Semantic] layer={state.current_layer.value} | "
                      f"msg='{last_msg[:40]}' | conf={sem_conf:.0%}")

                if sem_conf >= SEMANTIC_THRESHOLD:
                    ls = state.current_layer_state
                    if sem_conf > ls.confidence:
                        ls.confidence        = sem_conf
                        ls.symptom_positive  = sem_conf >= LOW_CONFIDENCE
                        ls.symptom_uncertain = sem_conf < 0.6
                        ls.symptom_detail    = last_msg

                    # Re-check early exit after semantic update
                    if self.fsm.check_early_exit(state):
                        return self._trigger_report(state)

        # ── Generate Question (Hybrid) ─────────────────────────────────────────
        ls = state.current_layer_state
        q_en, q_ar = self.llm.generate_question(
            body_part        = state.body_part,
            layer            = state.current_layer,
            history          = state.history,
            preferred_lang   = state.preferred_lang,
            is_followup      = ls.symptom_positive and ls.followups > 0,
            is_clarification = ls.symptom_uncertain and not ls.symptom_positive,
            force_llm        = state.force_llm_question,
            session_id       = session_id,
        )
        state.history.append({"role": "doctor", "content_en": q_en, "content_ar": q_ar})

        return {
            "status":                "QUESTION",
            "session_status":        "ACTIVE",
            "layer":                 state.current_layer.value,
            "layer_index":           state.layer_index,
            "previous_layer":        state.previous_layer,
            "remove_previous_layer": state.remove_previous_layer,
            "question_id":           len(state.history),
            "content_en":            q_en,
            "content_ar":            q_ar,
            "user_lang":             state.preferred_lang,
        }

print("Medical Agent loaded.")

## Cell 7 — LangGraph Multi-Agent Architecture *(Fix 4)*

Replaces the monolithic `MedicalAgent.process()` with a 5-node graph:

```
START → [triage_node] → [semantic_node] → [question_node] → END
                      ↘ [emergency_node]  [report_node]   → END
```

| Node | Responsibility |
|------|----------------|
| `triage_node` | FSM keyword detection + smart jump + layer state update |
| `semantic_node` | LLM semantic check when keywords find nothing |
| `question_node` | Generate next clinical question (Fixes 1-3 active) |
| `report_node` | Trigger async background report generation |
| `emergency_node` | Handle critical symptom keywords immediately |

Requires: `pip install langgraph` (handled automatically below)

In [ ]:
# Cell 7 — LangGraph Multi-Agent Architecture (Fix 4)
# Fix 4 — LangGraph Multi-Agent Architecture
# Install dependency first (run pip install if not present)

try:
    from langgraph.graph import StateGraph, END as LG_END
    print("langgraph already installed.")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "langgraph"], check=True)
    from langgraph.graph import StateGraph, END as LG_END
    print("langgraph installed.")

from typing import TypedDict, Literal, Optional as Opt
import copy, threading

# ── Shared Graph State ─────────────────────────────────────────────────────────

class GraphState(TypedDict):
    session_id:   str
    body_part:    str
    user_message: str
    session:      object                  # SessionState instance
    action:       Opt[str]                # set by triage_node
    sem_conf:     float                   # set by semantic_node
    response:     Opt[dict]               # final response to return to client
    llm:          object                  # LLMGenerator instance
    fsm:          object                  # LayerStateMachine instance


# ── Node 1: Triage ─────────────────────────────────────────────────────────────

def triage_node(state: GraphState) -> GraphState:
    """
    Runs the FSM keyword detection and smart jump logic.
    Sets state['action'] to: emergency / generate_report / advance_layer / ask_question
    """
    sess = state["session"]
    msg  = state["user_message"]
    fsm  = state["fsm"]

    # Reset per-turn flags
    sess.remove_previous_layer = False
    sess.force_llm_question    = False
    sess.last_keyword_matches  = 0

    action = fsm.decide(sess, msg)
    state["action"] = action.value
    print(f"[TriageNode] session={state['session_id']} | action={action.value} | layer={sess.current_layer.value}")
    return state


# ── Node 2: Semantic Classifier ────────────────────────────────────────────────

def semantic_node(state: GraphState) -> GraphState:
    """
    Only runs when keywords found nothing (last_keyword_matches == 0).
    Uses LLM to semantically validate whether the patient's message
    belongs to the current layer — fills the gap between keyword and full NLU.
    """
    sess = state["session"]
    llm  = state["llm"]

    # Only run semantic check when needed
    if not (state["action"] == "ask_question"
            and sess.last_keyword_matches == 0
            and sess.force_llm_question):
        state["sem_conf"] = 0.0
        return state

    last_msg = sess.history[-1].get("content_en", "")
    if not last_msg:
        state["sem_conf"] = 0.0
        return state

    sem_conf = llm.semantic_detect(last_msg, sess.current_layer)
    state["sem_conf"] = sem_conf
    print(f"[SemanticNode] layer={sess.current_layer.value} | conf={sem_conf:.0%}")

    if sem_conf >= SEMANTIC_THRESHOLD:
        ls = sess.current_layer_state
        if sem_conf > ls.confidence:
            ls.confidence        = sem_conf
            ls.symptom_positive  = sem_conf >= LOW_CONFIDENCE
            ls.symptom_uncertain = sem_conf < 0.6
            ls.symptom_detail    = last_msg

        # Re-check early exit after semantic update
        if state["fsm"].check_early_exit(sess):
            state["action"] = "generate_report"
            sess.session_status = "COMPLETED"

    return state


# ── Node 3: Question Generator ────────────────────────────────────────────────

def question_node(state: GraphState) -> GraphState:
    """
    Generates the next clinical question using Fixes 1-3:
    - Delimiter-based extraction (Fix 1)
    - Context-aware fallback (Fix 2)
    - History summarization (Fix 3)
    """
    sess = state["session"]
    llm  = state["llm"]
    fsm  = state["fsm"]

    # Advance layer if needed
    if state["action"] == "advance_layer":
        sess.previous_layer        = sess.current_layer.value
        sess.remove_previous_layer = True
        sess.layer_index          += 1
        sess.force_llm_question    = False

    ls   = sess.current_layer_state
    q_en, q_ar = llm.generate_question(
        body_part        = sess.body_part,
        layer            = sess.current_layer,
        history          = sess.history,
        preferred_lang   = sess.preferred_lang,
        is_followup      = ls.symptom_positive and ls.followups > 0,
        is_clarification = ls.symptom_uncertain and not ls.symptom_positive,
        force_llm        = sess.force_llm_question,
        session_id       = sess.session_id,
    )
    sess.history.append({"role": "doctor", "content_en": q_en, "content_ar": q_ar})

    state["response"] = {
        "status":                "QUESTION",
        "session_status":        "ACTIVE",
        "layer":                 sess.current_layer.value,
        "layer_index":           sess.layer_index,
        "previous_layer":        sess.previous_layer,
        "remove_previous_layer": sess.remove_previous_layer,
        "question_id":           len(sess.history),
        "content_en":            q_en,
        "content_ar":            q_ar,
        "user_lang":             sess.preferred_lang,
        "_node":                 "question_node",    # debug label
    }
    print(f"[QuestionNode] layer={sess.current_layer.value} | q_en={q_en[:60]}...")
    return state


# ── Node 4: Report Generator ──────────────────────────────────────────────────

def report_node(state: GraphState) -> GraphState:
    """Triggers async report generation and returns PROCESSING status."""
    sess     = state["session"]
    llm      = state["llm"]
    fsm      = state["fsm"]

    sess.is_complete    = True
    sess.session_status = "COMPLETED"
    severity, corr      = fsm.compute_severity(sess.layers)

    reports_db[sess.session_id] = {"status": "PROCESSING"}

    threading.Thread(
        target=_run_report_in_background,
        args=(
            sess.session_id, sess.body_part,
            copy.deepcopy(sess.layers), copy.deepcopy(sess.history),
            severity, corr, llm,
        ),
        daemon=True,
    ).start()

    state["response"] = {
        "status":         "PROCESSING",
        "session_status": "COMPLETED",
        "session_id":     sess.session_id,
        "message":        "Report is being generated. Poll /get_report/{session_id} every 10 seconds.",
        "_node":          "report_node",
    }
    print(f"[ReportNode] session={sess.session_id} | severity={severity}")
    return state


# ── Node 5: Emergency Handler ─────────────────────────────────────────────────

def emergency_node(state: GraphState) -> GraphState:
    """Returns emergency response immediately."""
    sess = state["session"]
    sess.is_complete    = True
    sess.session_status = "EMERGENCY"

    state["response"] = {
        "status":         "EMERGENCY",
        "session_status": "EMERGENCY",
        "message_en":     "Your symptoms may indicate a life-threatening emergency. "
                          "Call emergency services (123) or go to the nearest ER immediately.",
        "message_ar":     "قد تشير أعراضك إلى حالة طارئة. "
                          "اتصل بالطوارئ (123) أو اذهب فوراً لأقرب غرفة طوارئ.",
        "_node":          "emergency_node",
    }
    print(f"[EmergencyNode] session={sess.session_id}")
    return state


# ── Routing Functions ─────────────────────────────────────────────────────────

def route_after_triage(state: GraphState) -> Literal["semantic_node","question_node","report_node","emergency_node"]:
    action = state["action"]
    if action == "emergency":            return "emergency_node"
    if action == "generate_report":      return "report_node"
    if action in ("ask_question","advance_layer"): return "semantic_node"
    return "question_node"

def route_after_semantic(state: GraphState) -> Literal["question_node","report_node"]:
    return "report_node" if state["action"] == "generate_report" else "question_node"


# ── Build the Graph ────────────────────────────────────────────────────────────

def build_medical_graph():
    g = StateGraph(GraphState)
    g.add_node("triage_node",    triage_node)
    g.add_node("semantic_node",  semantic_node)
    g.add_node("question_node",  question_node)
    g.add_node("report_node",    report_node)
    g.add_node("emergency_node", emergency_node)

    g.set_entry_point("triage_node")
    g.add_conditional_edges("triage_node",   route_after_triage)
    g.add_conditional_edges("semantic_node", route_after_semantic)
    g.add_edge("question_node",  LG_END)
    g.add_edge("report_node",    LG_END)
    g.add_edge("emergency_node", LG_END)

    return g.compile()

medical_graph = build_medical_graph()
print("✅ Fix 4 applied: LangGraph multi-agent graph compiled.")
print("   Nodes: triage → semantic → question / report / emergency")


# ── Patch MedicalAgent.process to use the graph ───────────────────────────────

def _graph_process(self, session_id: str, body_part: str, user_message: str) -> dict:
    """
    Drop-in replacement for MedicalAgent.process() using the LangGraph pipeline.
    The session state management (get_or_create, history append, language detect)
    is preserved exactly so the API layer needs no changes.
    """
    state_obj = self._get_or_create(session_id, body_part)

    # Already complete — return stored report
    if state_obj.is_complete:
        entry = reports_db.get(session_id)
        if entry: return entry
        saved = load_report_from_disk(session_id)
        if saved: return {"status": "COMPLETED", "report": saved}
        return {"status": "PROCESSING", "message": "Poll /get_report/{session_id}."}

    # Language detection on first real message
    if len(state_obj.history) <= 1:
        state_obj.preferred_lang = detect_language(user_message)

    state_obj.history.append({
        "role":       "patient",
        "content_en": user_message,
        "content_ar": user_message,
        "lang":       detect_language(user_message),
    })

    # Run through LangGraph
    graph_input: GraphState = {
        "session_id":   session_id,
        "body_part":    body_part,
        "user_message": user_message,
        "session":      state_obj,
        "action":       None,
        "sem_conf":     0.0,
        "response":     None,
        "llm":          self.llm,
        "fsm":          self.fsm,
    }
    final = medical_graph.invoke(graph_input)
    return final["response"]


MedicalAgent.process = _graph_process
print("   MedicalAgent.process() now routes through the LangGraph pipeline.")
print("\n🚀 All 4 fixes active. Restart the session with /start and test the chatbot.")


## Cell 8 — FastAPI Server & Launch

In [ ]:
# Cell 8 — FastAPI Server & Launch
# Cell 7 — FastAPI Server Assembly & Launch

class StartRequest(BaseModel):
    body_part:       str
    chief_complaint: Optional[str] = None   # optional — enables smart layer entry

class ChatRequest(BaseModel):
    session_id: str
    body_part:  str
    message:    str

agent: Optional[MedicalAgent] = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global agent
    print("[Lifespan] Initializing Medical Agent...")
    try:
        llm   = LLMGenerator(MODEL_ID)
        agent = MedicalAgent(llm)
        print("[Lifespan] Agent ready and accepting connections.")
        yield
    finally:
        agent = None
        print("[Lifespan] Agent shut down.")

app = FastAPI(title="Medical AI Backend", version="2.0", lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Endpoints ──────────────────────────────────────────────────────────────────

@app.get("/health")
async def health():
    return {"status": "ok", "agent_ready": agent is not None}


@app.post("/start")
async def start_session(req: StartRequest):
    """
    Start a new diagnostic session and receive the first clinical question.

    Request:
      body_part        (required) — e.g. "knee", "hand", "lower_back"
      chief_complaint  (optional) — patient's complaint in their own words.
                       When provided, the agent skips irrelevant layers automatically.

    Response (unchanged structure):
      session_id, first_question_en, first_question_ar, layer, layer_index
    """
    if not agent:
        raise HTTPException(status_code=503, detail="Agent not ready")
    return agent.start_session(req.body_part, req.chief_complaint)


@app.post("/chat")
async def chat(req: ChatRequest):
    """
    Send a patient message and receive the next question or report trigger.

    Request:  session_id, body_part, message

    Response status values:
      QUESTION   — next clinical question (content_en, content_ar, layer, ...)
      PROCESSING — report started in background, poll /get_report/{session_id}
      EMERGENCY  — critical symptoms detected (message_en, message_ar)
    """
    if not agent:
        raise HTTPException(status_code=503, detail="Agent not ready")
    return agent.process(req.session_id, req.body_part, req.message)


@app.get("/get_report/{session_id}")
async def get_report(session_id: str):
    """
    Poll this endpoint every 10 seconds after receiving status=PROCESSING from /chat.

    Responses:
      {status: PROCESSING}                        — still generating, try again
      {status: COMPLETED, report: {...}}          — report ready
      {status: FAILED,    error: "..."}           — generation error

    Report fields (always present):
      diagnosis_en/ar, severity, assessment_confidence,
      specialist_en/ar, immediate_actions_en/ar,
      lifestyle_advice_en/ar, differentials, red_flags,
      follow_up_en/ar, summary_en/ar, correlation_alerts (if any)

    On COMPLETED: session is automatically cleaned from memory.
    If server restarted between generation and retrieval — falls back to disk.
    """
    if not agent:
        raise HTTPException(status_code=503, detail="Agent not ready")

    entry = reports_db.get(session_id)

    if not entry:
        saved = load_report_from_disk(session_id)
        if saved:
            entry = {"status": "COMPLETED", "report": saved}
            reports_db[session_id] = entry

    if not entry:
        raise HTTPException(status_code=404, detail="Session not found or report not started")

    # Clean from memory once client confirms receipt
    if entry["status"] == "COMPLETED":
        agent.cleanup_session(session_id)

    return entry


# ── Server Launch ──────────────────────────────────────────────────────────────

def run_production_server():
    print("[Cleanup] Checking port 8000...")
    try: os.system(f"fuser -k {PORT}/tcp")
    except Exception: pass

    try:
        ngrok.kill()
        if NGROK_AUTH_TOKEN:
            ngrok.set_auth_token(NGROK_AUTH_TOKEN)
            tunnel = ngrok.connect(PORT)
            print("=" * 55)
            print(f"PUBLIC API : {tunnel.public_url}")
            print(f"SWAGGER    : {tunnel.public_url}/docs")
            print(f"HEALTH     : {tunnel.public_url}/health")
            print("=" * 55)
    except Exception as e:
        print(f"[Ngrok] Error: {e}")

    def _start():
        uvicorn.Server(
            uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
        ).run()

    threading.Thread(target=_start, daemon=True).start()
    print("Server started.")

run_production_server()